# init

> Router assembly for Phase 2 alignment routes

In [ ]:
#| default_exp routes.init

In [ ]:
#| export
from typing import List, Dict, Callable, Tuple

from cjm_fasthtml_app_core.core.routing import APIRouter

from cjm_fasthtml_card_stack.core.models import CardStackUrls

from cjm_transcript_vad_align.models import AlignmentUrls
from cjm_transcript_vad_align.services.alignment import AlignmentService
from cjm_transcript_vad_align.routes.core import WorkflowStateStore
from cjm_transcript_vad_align.routes.card_stack import init_card_stack_router
from cjm_transcript_vad_align.routes.handlers import init_workflow_router
from cjm_transcript_vad_align.routes.audio import init_audio_router
from cjm_transcript_source_select.services.source import SourceService

## Router Assembly

In [ ]:
#| export
def init_alignment_routers(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    source_service:SourceService,  # Service for fetching source blocks
    alignment_service:AlignmentService,  # Service for VAD analysis
    prefix:str,  # Base prefix for alignment routes (e.g., "/workflow/align")
    audio_src_url:str,  # URL for audio_src route (from core router)
    wrapped_init:Callable=None,  # Optional wrapped init handler
) -> Tuple[List[APIRouter], AlignmentUrls, Dict[str, Callable]]:  # (routers, urls, merged_routes)
    """Initialize and return all alignment routers with URL bundle."""
    # Create empty URL bundle (populated after routes are defined)
    urls = AlignmentUrls()

    # Initialize focused routers
    card_stack_router, card_stack_routes = init_card_stack_router(
        state_store, workflow_id, f"{prefix}/card_stack", urls
    )
    workflow_router, workflow_routes = init_workflow_router(
        state_store, workflow_id, source_service, alignment_service,
        f"{prefix}/workflow", urls,
        handler_init=wrapped_init,
    )
    audio_router, audio_routes = init_audio_router(
        state_store, workflow_id, f"{prefix}/audio", urls
    )

    # Populate the URL bundle using .to() on route functions
    urls.card_stack = CardStackUrls(
        nav_up=card_stack_routes["nav_up"].to(),
        nav_down=card_stack_routes["nav_down"].to(),
        nav_first=card_stack_routes["nav_first"].to(),
        nav_last=card_stack_routes["nav_last"].to(),
        nav_page_up=card_stack_routes["nav_page_up"].to(),
        nav_page_down=card_stack_routes["nav_page_down"].to(),
        nav_to_index=card_stack_routes["nav_to_index"].to(),
        update_viewport=card_stack_routes["update_viewport"].to(),
        save_width=card_stack_routes["save_width"].to(),
    )
    urls.init = workflow_routes["init"].to()
    urls.audio_src = audio_src_url
    urls.toggle_auto_nav = audio_routes["toggle_auto_nav"].to()
    urls.speed_change = audio_routes["speed_change"].to()

    # Merge all routes for external access
    merged_routes = {
        **card_stack_routes,
        **workflow_routes,
        **audio_routes,
    }

    routers = [card_stack_router, workflow_router, audio_router]

    return routers, urls, merged_routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()